In [ ]:
from rich import print
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model

llm=init_chat_model(model="groq:qwen/qwen3-32b")

#### Tools

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()

In [ ]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def multiply(a:int, b:int)->int:
    """Multiply two numbers"""
    return a*b

@tool
def add(a:int, b:int)->int:
    """Add two numbers"""
    return a+b

@tool
def current_time()->str:
    """Get current system time"""
    return datetime.now().strftime("%I:%M %p")

In [ ]:
tools=[multiply,add,current_time,tavily]

In [ ]:
llm_with_tools=llm.bind_tools(tools=tools)

In [ ]:
tools_by_name={
    tool.name: tool for tool in tools
}

In [ ]:
print(tools_by_name)

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list,add_messages]

In [ ]:
from langchain_core.messages import SystemMessage

def chatbot_node(state:AgentState):
    messages=state['messages']
    response = llm_with_tools.invoke(
        [
            SystemMessage(
                content="""You are a helpful assistant. Use tools whenever needed"""
            )
        ]
        + messages
    )

    return {"messages":[response]}

In [ ]:
from langchain_core.messages import ToolMessage

def tool_node(state:AgentState):
    messages=state["messages"]
    last_message=messages[-1]
    tool_messages=[]

    for tool_call in last_message.tool_calls:
        tool_name=tool_call["name"]
        tool_args=tool_call["args"]
        tool=tools_by_name[tool_name]
        result=tool.invoke(tool_args)
        tool_message=ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
        tool_messages.append(tool_message)

    return {"messages":tool_messages}

In [ ]:
from langgraph.graph import END

def should_continue(state:AgentState):
    messages=state["messages"]
    last_message=messages[-1]

    if hasattr(last_message,"tool_calls") and last_message.tool_calls:
        return "tool_node"
    
    return END

In [ ]:
from langgraph.graph import StateGraph

graph=StateGraph(AgentState)

graph.add_node("chatbot_node",chatbot_node)
graph.add_node("tool_node",tool_node)

graph.set_entry_point("chatbot_node")
graph.add_conditional_edges(
    "chatbot_node",
    should_continue
)
graph.add_edge("tool_node","chatbot_node")

app=graph.compile()

In [ ]:
app

In [ ]:
from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [HumanMessage(content="How much 9 times 8")]
})

In [ ]:
print(result)

---

In [ ]:
from dotenv import load_dotenv
from rich import print
from datetime import datetime

# =========================================================
# LOAD ENV
# =========================================================

load_dotenv()

# =========================================================
# LLM
# =========================================================

from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="groq:qwen/qwen3-32b",
    temperature=0
)

# =========================================================
# TOOLS
# =========================================================

from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


@tool
def current_time() -> str:
    """Get current system time"""
    return datetime.now().strftime("%I:%M %p")


search_tool = TavilySearchResults(max_results=3)

tools = [
    multiply,
    add,
    current_time,
    search_tool,
]

# =========================================================
# BIND TOOLS
# =========================================================

llm_with_tools = llm.bind_tools(tools)

# =========================================================
# GRAPH STATE
# =========================================================

from langgraph.graph import MessagesState

# MessagesState already provides:
# {
#    "messages": Annotated[list, add_messages]
# }

# =========================================================
# CHATBOT NODE
# =========================================================

from langchain_core.messages import SystemMessage

def assistant_node(state: MessagesState):

    system_prompt = SystemMessage(
        content="""
        You are a helpful AI assistant.

        Use:
        - Tavily for web searches/current events
        - math tools for calculations
        - current_time for time queries

        Be concise and accurate.
        """
    )

    response = llm_with_tools.invoke(
        [system_prompt] + state["messages"]
    )

    return {
        "messages": [response]
    }

# =========================================================
# PREBUILT TOOL NODE
# =========================================================

from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

# =========================================================
# CONDITIONAL ROUTING
# =========================================================

from langgraph.prebuilt import tools_condition

# =========================================================
# GRAPH
# =========================================================

from langgraph.graph import StateGraph, START, END

graph = StateGraph(MessagesState)

graph.add_node("assistant", assistant_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "assistant")

graph.add_conditional_edges(
    "assistant",
    tools_condition,
    {
        "tools": "tools",
        END: END,
    }
)

graph.add_edge("tools", "assistant")

# =========================================================
# MEMORY / CHECKPOINTING
# =========================================================

from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

app = graph.compile(
    checkpointer=memory
)

# =========================================================
# CHAT LOOP
# =========================================================

from langchain_core.messages import HumanMessage

config = {
    "configurable": {
        "thread_id": "demo-user"
    }
}

print("\n[bold green]AI Chatbot Started[/bold green]")
print("Type 'exit' to quit\n")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    result = app.invoke(
        {
            "messages": [
                HumanMessage(content=user_input)
            ]
        },
        config=config
    )

    print("\n[bold cyan]Bot:[/bold cyan]")
    print(result["messages"][-1].content)
    print()